In [ ]:
%pip install google-adk google-generativeai -q

# --- Import all necessary libraries ---
import os
import sys
import json
import asyncio
import random
import string

from uuid import uuid4
from typing import Any, List

import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, Markdown, display

# --- ADK, Agent, and Evaluation Components ---
from google.adk.agents import Agent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search
from google.adk.sessions import InMemorySessionService, Session
from google.genai import types
from google.genai.types import Content, Part

from dotenv import load_dotenv


print(" All libraries are ready to go!")



 All libraries are ready to go!


In [13]:
# --- API Key Configuration ---
from google.colab import userdata

# Option 1: Use Colab Secrets (recommended)
# Go to the 🔑 icon in the left sidebar, add a secret named GOOGLE_API_KEY
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    print("✅ API key loaded from Colab Secrets.")
except Exception:
    # Option 2: Paste it directly (less secure but fine for learning)
    import getpass
    GOOGLE_API_KEY = getpass.getpass("🔑 Enter your Google AI Studio API key: ")
    print(" API key entered manually.")

 API key entered manually.


In [15]:
# --- Set Environment Variables for ADK ---

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"

print(f" API key configured (starts with '{GOOGLE_API_KEY[:6]}...')")
print(" Using Google AI Studio (not Vertex AI).")


 API key configured (starts with 'AQ.Ab8...')
 Using Google AI Studio (not Vertex AI).


In [17]:
def create_day_trip_agent():
    """Create the Spontaneous Day Trip Generator agent"""
    return Agent(
        name="day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent specialized in generating spontaneous full-day itineraries based on mood, interests, and budget.",
        instruction="""
        You are the "Spontaneous Day Trip" Generator 🚗 - a specialized AI assistant that creates engaging full-day itineraries.

        Your Mission:
        Transform a simple mood or interest into a complete day-trip adventure with real-time details, while respecting a budget.

        Guidelines:
        1. **Budget-Aware**: Pay close attention to budget hints like 'cheap', 'affordable', or 'splurge'. Use Google Search to find activities (free museums, parks, paid attractions) that match the user's budget.
        2. **Full-Day Structure**: Create morning, afternoon, and evening activities.
        3. **Real-Time Focus**: Search for current operating hours and special events.
        4. **Mood Matching**: Align suggestions with the requested mood (adventurous, relaxing, artsy, etc.).

        RETURN itinerary in MARKDOWN FORMAT with clear time blocks and specific venue names.
        """,
        tools=[google_search]
    )

day_trip_agent = create_day_trip_agent()
print(f"🧞 Agent '{day_trip_agent.name}' is created and ready for adventure!")

🧞 Agent 'day_trip_agent' is created and ready for adventure!


In [56]:
# --- A Helper Function to Run Our Agents ---
# We'll use this function throughout the notebook to make running queries easy.

async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str, is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if not is_router:
                # Let's see what the agent is thinking!
                print(f"EVENT: {event}")
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
     print("\n" + "-"*50)
     print("✅ Final Response:")
     display(Markdown(final_response))
     print("-"*50 + "\n")

    return final_response

# --- Initialize our Session Service ---
# This one service will manage all the different sessions in our notebook.
session_service = InMemorySessionService()
my_user_id = "adk_adventurer_001"

In [57]:
# --- Let's test the Day Trip Genie! ---

async def run_day_trip_genie():
    # Create a new, single-use session for this query
    day_trip_session = await session_service.create_session(
        app_name=day_trip_agent.name,
        user_id=my_user_id
    )

    # Note the new budget constraint in the query!
    query = "Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!"
    print(f"🗣️ User Query: '{query}'")

    await run_agent_query(day_trip_agent, query, day_trip_session, my_user_id)

await run_day_trip_genie()

🗣️ User Query: 'Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!'

🚀 Running query for agent: 'day_trip_agent' in session: '99ffd3ff-3d04-4e51-b475-51c2706a7cf0'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Here's a relaxing and artsy day trip itinerary near Sunnyvale, California, designed to be affordable and enjoyable!

***

## Relaxing & Artsy Day Trip near Sunnyvale, CA (Affordable)

This itinerary combines free art experiences with serene natural beauty and an affordable dining option.

### 🚗 **Morning (10:00 AM - 1:00 PM): Immerse in Art & Sculpture**

*   **Destination:** **Cantor Arts Center at Stanford University** (328 Lomita Dr, Stanford, CA 94305)
*   **Cost:** Free admission, free parking on weekends.
*   **Activity:** Start your day with a visit to the renowned Cantor Arts Center, home to a diverse collection spanning 5,000 years of art, including a significant collection of Rodin bronze sculptures

Here's a relaxing and artsy day trip itinerary near Sunnyvale, California, designed to be affordable and enjoyable!

***

## Relaxing & Artsy Day Trip near Sunnyvale, CA (Affordable)

This itinerary combines free art experiences with serene natural beauty and an affordable dining option.

### 🚗 **Morning (10:00 AM - 1:00 PM): Immerse in Art & Sculpture**

*   **Destination:** **Cantor Arts Center at Stanford University** (328 Lomita Dr, Stanford, CA 94305)
*   **Cost:** Free admission, free parking on weekends.
*   **Activity:** Start your day with a visit to the renowned Cantor Arts Center, home to a diverse collection spanning 5,000 years of art, including a significant collection of Rodin bronze sculptures displayed both indoors and in the outdoor Rodin Sculpture Garden. Wander through the galleries at your own pace, taking in the various exhibitions. The serene setting of Stanford's campus also offers a relaxing atmosphere. The museum is open Wednesday and Friday from 11:00 a.m.–6:00 p.m.; Thursday from 11:00 a.m.–8:00 p.m.; and Saturday and Sunday from 10:00 a.m.–5:00 p.m.

### 🥪 **Lunch (1:00 PM - 2:00 PM): Affordable Picnic**

*   **Destination:** **Stanford University Oval or nearby park**
*   **Cost:** Bring your own picnic lunch or grab an affordable bite from a cafe on campus.
*   **Activity:** Enjoy a leisurely picnic lunch on the spacious Oval at Stanford University, or find a quiet bench amidst the beautiful architecture and landscaping. Alternatively, Tootsie's at Cantor, the on-site cafe, offers coffee/tea services, lunch service, and tea service during museum hours.

### 🌸 **Afternoon (2:30 PM - 5:30 PM): Serenity in a Japanese Garden**

*   **Destination:** **Japanese Friendship Garden (within Kelley Park)** (1300 Senter Rd, San Jose, CA 95112)
*   **Cost:** Free admission, $6 for parking.
*   **Activity:** Drive a short distance to San Jose and discover the tranquil beauty of the Japanese Friendship Garden. Patterned after Okayama's Korakuen Garden, this six-acre oasis features picturesque Japanese bridges, cascading waterfalls, and ponds stocked with golden Koi fish. Stroll along the winding paths, find a peaceful spot to sit, and enjoy the calming sounds and sights. The garden is typically open daily from 10:00 a.m. to 7:00 p.m. during the summer season.

### 🍽️ **Evening (6:30 PM onwards): Casual & Affordable Dinner**

*   **Destination:** **Metro City Restaurant & Bar** (155 S Murphy Ave, Sunnyvale, CA 94086)
*   **Cost:** Affordable
*   **Activity:** Head back towards Sunnyvale for a casual yet pleasant dinner. Metro City Restaurant & Bar in downtown Sunnyvale is known for its affordable prices, generous portions, and casual atmosphere, with both indoor and outdoor seating options. They offer breakfast, lunch, and dinner all day. Another highly-rated and affordable option is **Madras Cafe** (1177 W El Camino Real, Sunnyvale, CA 94087) for a tasty vegetarian South Indian meal.

Enjoy your relaxing and artsy day trip!

--------------------------------------------------

